In [1]:
import pandas as pd 
df = pd.read_csv('student.csv')
df.head() 



,hours_study,attendance,marks
0,2,60,40
1,3,65,45
2,4,70,50
3,5,75,55
4,6,80,65


In [2]:
import pandas as pd
import sagemaker
from sagemaker import get_execution_role

# Load dataset
df = pd.read_csv('student.csv')

# Features (X) and Target (y)
X = df[['hours_study', 'attendance']]
y = df['marks']

# Combine into one dataset (SageMaker expects target first)
train_df = pd.concat([y, X], axis=1)

train_df.to_csv('train.csv', index=False, header=False)

print(train_df.head())

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
   marks  hours_study  attendance
0     40            2          60
1     45            3          65
2     50            4          70
3     55            5          75
4     65            6          80


In [3]:
session = sagemaker.Session()
bucket = session.default_bucket()
prefix = 'student-model'

s3_path = session.upload_data(path='train.csv', bucket=bucket, key_prefix=prefix)

print("S3 Path:", s3_path)

S3 Path: s3://sagemaker-ap-south-1-133074138303/student-model/train.csv


In [4]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

role = get_execution_role()

container = sagemaker.image_uris.retrieve(
    framework='xgboost',
    region=session.boto_region_name,
    version='1.5-1'
)

estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=session
)

estimator.set_hyperparameters(
    objective='reg:squarederror',
    num_round=100
)

train_input = TrainingInput(
    s3_data=s3_path,
    content_type='text/csv'
)

estimator.fit({'train': train_input})

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-04-20-12-54-17-865


2026-04-20 12:54:18 Starting - Starting the training job...
2026-04-20 12:54:32 Starting - Preparing the instances for training...
2026-04-20 12:54:58 Downloading - Downloading input data...
2026-04-20 12:55:43 Downloading - Downloading the training image......
2026-04-20 12:56:44 Training - Training image download completed. Training in progress./miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-04-20 12:56:45.707 ip-10-0-243-40.ap-south-1.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-04-20 12:56:45.731 ip-10-0-243-40.ap-south-1.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-04-20:12:56:46:INFO] Imported framework sagemaker_xgboost_container.training
[2026-04-20:12:56:46:INFO] Failed to parse hyperpar

In [5]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)

print("Endpoint Name:", predictor.endpoint_name)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-04-20-12-59-51-462
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-04-20-12-59-51-462
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2026-04-20-12-59-51-462


------!Endpoint Name: sagemaker-xgboost-2026-04-20-12-59-51-462


In [9]:
from sagemaker.serializers import CSVSerializer

predictor.serializer = CSVSerializer()

import numpy as np
test_data = np.array([[5, 80]])

result = predictor.predict(test_data)
print("Prediction:", result)

Prediction: b'65.54659271240234\n'


In [10]:
import boto3

runtime = boto3.client('sagemaker-runtime')

endpoint_name = predictor.endpoint_name

payload = "5,80"

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='text/csv',
    Body=payload
)

result = response['Body'].read().decode()
print("API Result:", result)

API Result: 65.54659271240234



FOR LAMBDA FUNCTION USE THE FOLLWING CODE

POLICIES:
1. SAGEMAKERFULLACCESS
2. S3FULLACCESS
3. LAMBDABASICEXECUTION


GO TO CONFIGURATION AND THEN ENVIRONMENT VARIALBS AND ADD THE FOLLWING :
ENDPOINT_NAME : "ENDPOINT NAME"

FOR TESTING USE:
IN LAMBDA FUNCTION TESING:
{
  "body": "{\"hours_study\": 5, \"attendance\": 80}"
}


IN REQBIN OR POSTMAN:

{
  "hours_study": 5, "attendance": 80
}



In [ ]:
import json
import boto3
import os

# Create SageMaker runtime client
runtime = boto3.client('sagemaker-runtime')

# Store endpoint name as environment variable (recommended)
ENDPOINT_NAME = os.environ.get("ENDPOINT_NAME")

def lambda_handler(event, context):
    try:
        # Parse input (API Gateway sends JSON)
        body = json.loads(event.get("body", "{}"))
        
        hours = body.get("hours_study")
        attendance = body.get("attendance")

        # Validate input
        if hours is None or attendance is None:
            return {
                "statusCode": 400,
                "body": json.dumps({"error": "Missing input values"})
            }

        # Prepare payload for SageMaker
        payload = f"{hours},{attendance}"

        # Invoke endpoint
        response = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='text/csv',
            Body=payload
        )

        result = response['Body'].read().decode()

        return {
            "statusCode": 200,
            "body": json.dumps({
                "predicted_marks": result
            })
        }

    except Exception as e:
        return {
            "statusCode": 500,
            "body": json.dumps({"error": str(e)})
        }